# LLM Infrastructure Finance Evaluation Assistant

This is an original, GitHub-ready notebook inspired by the type of work that could be useful in AI infrastructure finance, model evaluation, and strategy roles.

It does **not** copy course lab text, outputs, screenshots, or datasets.  
It uses synthetic AI infrastructure finance cases and demonstrates a complete LLM workflow.

## What this notebook covers

- Synthetic finance / AI infrastructure dataset creation with Hugging Face `datasets`
- Local text generation with Hugging Face `transformers`
- Retrieval-augmented prompting with `sentence-transformers`
- ROUGE evaluation with Hugging Face `evaluate`
- Structured output validation
- Token, latency, and rough cost tracking
- PEFT / LoRA setup using Hugging Face `peft`
- Optional OpenAI and Anthropic API wrappers
- Executive summary generation

## Use case

Given an AI infrastructure investment request, generate an analyst-style response that covers:

1. Business objective
2. Capacity / compute need
3. Cost drivers
4. Risks
5. Recommended next step


In [ ]:
# Install dependencies.
# In GitHub, this cell will display as code. In Colab/Jupyter, run it once.

%pip install -q \
    torch \
    transformers \
    datasets \
    evaluate \
    accelerate \
    peft \
    sentence-transformers \
    sentencepiece \
    protobuf \
    rouge_score \
    pandas \
    numpy \
    scikit-learn \
    openai \
    anthropic


In [ ]:
import os
import re
import json
import time
import math
import textwrap
from dataclasses import dataclass
from typing import Dict, List, Optional, Any

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    GenerationConfig,
)

import evaluate

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from peft import LoraConfig, TaskType, get_peft_model


## 1. Create a synthetic AI infrastructure finance dataset

The dataset is intentionally synthetic. It represents common request types an AI finance / strategy / infrastructure team might evaluate:

- Training cluster expansion
- Inference cost reduction
- Enterprise deployment
- Evaluation and safety infrastructure
- Cloud vs reserved capacity tradeoff


In [ ]:
synthetic_cases = [
    {
        "case_id": "TRAIN-001",
        "request_type": "training_capacity",
        "request": (
            "Research wants to reserve an additional 256 H100 GPUs for eight weeks "
            "to run frontier model pre-training experiments. The team expects higher model quality, "
            "but the finance team needs to understand utilization risk, opportunity cost, and budget impact."
        ),
        "policy_hint": "Large training requests require utilization assumptions, experiment gating, fallback plan, and milestone-based approval.",
        "reference_answer": (
            "The request should be evaluated as a staged training capacity investment. Key drivers are GPU weeks, utilization, "
            "networking and storage overhead, failed-run risk, and opportunity cost versus other research priorities. "
            "Approve only if the team defines experiment milestones, target model-quality metrics, and a fallback plan for unused capacity."
        ),
    },
    {
        "case_id": "INF-002",
        "request_type": "inference_cost",
        "request": (
            "Product wants to launch a new assistant feature to 10 million monthly active users. "
            "Expected usage is 18 requests per user per month, with 1,200 input tokens and 350 output tokens per request. "
            "Leadership needs a scalable inference cost framework before launch."
        ),
        "policy_hint": "High-volume inference launches require unit-cost modeling, routing strategy, caching, latency target, and cost guardrails.",
        "reference_answer": (
            "This should be modeled as a recurring inference COGS decision. The analysis should estimate input and output token volume, "
            "latency requirements, model routing, cache hit rate, and gross-margin impact. The launch should include cost guardrails, "
            "usage monitoring, and a fallback to smaller models for low-complexity requests."
        ),
    },
    {
        "case_id": "ENT-003",
        "request_type": "enterprise_deployment",
        "request": (
            "A strategic enterprise customer wants a private deployment with stricter data retention controls, "
            "audit logging, and custom evaluation reports. Sales wants to discount the contract to win the account."
        ),
        "policy_hint": "Enterprise deployments require security review, privacy constraints, support cost, custom eval cost, and minimum viable margin.",
        "reference_answer": (
            "The deal should be evaluated using incremental revenue, deployment cost, support burden, privacy/security requirements, and custom evaluation cost. "
            "Discounting may be justified only if the account meets margin thresholds or creates strategic learning value. The team should define non-standard costs before approval."
        ),
    },
    {
        "case_id": "EVAL-004",
        "request_type": "eval_infrastructure",
        "request": (
            "The safety team requests funding for a larger automated evaluation pipeline covering jailbreaks, hallucination, "
            "tool-use reliability, and domain-specific failure modes. The request does not directly generate revenue."
        ),
        "policy_hint": "Safety and eval infrastructure should be assessed as risk reduction, launch readiness, incident prevention, and governance capability.",
        "reference_answer": (
            "This should be treated as risk-reduction and launch-readiness infrastructure. The business case should connect eval coverage to lower incident risk, faster release gates, "
            "better model governance, and reduced manual review load. Funding can be justified if it becomes a reusable platform across launches."
        ),
    },
    {
        "case_id": "RES-005",
        "request_type": "capacity_commitment",
        "request": (
            "Infrastructure proposes a one-year reserved capacity commitment to reduce unit cost. "
            "The commitment improves pricing but creates risk if model roadmap, traffic, or hardware requirements change."
        ),
        "policy_hint": "Reserved capacity decisions require demand forecast, utilization floor, flexibility value, break-even analysis, and exit risk.",
        "reference_answer": (
            "The decision should compare discounted reserved capacity against on-demand flexibility. Key inputs are expected utilization, demand volatility, roadmap uncertainty, "
            "break-even volume, and downside risk from unused capacity. Approval should require a utilization floor and scenario analysis."
        ),
    },
]

dataset = Dataset.from_list(synthetic_cases)
df_cases = dataset.to_pandas()
df_cases


## 2. Load a Hugging Face model with `transformers`

This notebook uses `google/flan-t5-small` by default so it can run on CPU.  
For stronger results, change `MODEL_NAME` to `google/flan-t5-base` or another instruction-tuned model you can run locally.


In [ ]:
MODEL_NAME = os.getenv("HF_MODEL_NAME", "google/flan-t5-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Using model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)


In [ ]:
def count_tokens(text: str) -> int:
    return len(tokenizer(text)["input_ids"])


def generate_hf(
    prompt: str,
    max_new_tokens: int = 180,
    temperature: float = 0.0,
    do_sample: bool = False,
) -> Dict[str, Any]:
    """Generate text using the local Hugging Face model and return metadata."""
    
    generation_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
    )
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)
    
    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            generation_config=generation_config,
        )
    latency_seconds = time.time() - start
    
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    return {
        "output_text": output_text,
        "input_tokens": int(inputs["input_ids"].shape[-1]),
        "output_tokens": count_tokens(output_text),
        "latency_seconds": latency_seconds,
    }


## 3. Prompt templates

We compare three prompt styles:

1. Basic prompt
2. Finance analyst prompt
3. Retrieval-grounded prompt


In [ ]:
def basic_prompt(case: Dict[str, Any]) -> str:
    return f"""
Summarize this AI infrastructure finance request.

Request:
{case["request"]}

Summary:
"""


def finance_analyst_prompt(case: Dict[str, Any]) -> str:
    return f"""
You are an AI infrastructure finance analyst.

Analyze the request below and produce a concise finance review.

Request:
{case["request"]}

Your response must include:
- Business objective
- Compute or capacity need
- Main cost drivers
- Key risks
- Recommended next step
"""


def structured_prompt(case: Dict[str, Any], retrieved_policy: Optional[str] = None) -> str:
    policy_section = f"Relevant internal policy guidance:\n{retrieved_policy}\n" if retrieved_policy else ""
    
    return f"""
You are an AI infrastructure finance analyst supporting model deployment and capacity planning.

{policy_section}
Request:
{case["request"]}

Return a concise decision memo with these exact sections:
Business Objective:
Compute Need:
Cost Drivers:
Risks:
Recommended Next Step:
"""


In [ ]:
sample_case = synthetic_cases[1]

for name, prompt_fn in [
    ("basic", basic_prompt),
    ("finance_analyst", finance_analyst_prompt),
]:
    prompt = prompt_fn(sample_case)
    result = generate_hf(prompt)
    print("=" * 100)
    print(f"Prompt type: {name}")
    print("-" * 100)
    print(result["output_text"])
    print()
    print({
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "latency_seconds": round(result["latency_seconds"], 3),
    })


## 4. Mini RAG with `sentence-transformers`

This section creates a tiny synthetic policy knowledge base and retrieves the most relevant policy for each request.


In [ ]:
policy_docs = [
    {
        "policy_id": "POL-TRAINING-CAPACITY",
        "text": (
            "Large model training capacity requests must include GPU count, duration, expected utilization, "
            "experiment milestones, quality metrics, fallback plan, storage and networking assumptions, and opportunity cost."
        ),
    },
    {
        "policy_id": "POL-INFERENCE-COGS",
        "text": (
            "High-volume inference launches must include token volume, input and output token assumptions, latency target, "
            "model routing strategy, cache hit rate, unit cost, gross-margin impact, and budget guardrails."
        ),
    },
    {
        "policy_id": "POL-ENTERPRISE-DEPLOYMENT",
        "text": (
            "Private enterprise deployments must include security review, privacy requirements, data retention policy, "
            "audit logging, customer support cost, custom evaluation scope, and minimum margin threshold."
        ),
    },
    {
        "policy_id": "POL-SAFETY-EVAL",
        "text": (
            "Safety and evaluation infrastructure should be assessed based on release readiness, risk reduction, incident prevention, "
            "manual review reduction, governance value, and reuse across launches."
        ),
    },
    {
        "policy_id": "POL-RESERVED-CAPACITY",
        "text": (
            "Reserved capacity commitments must include demand forecast, utilization floor, break-even analysis, downside scenario, "
            "hardware roadmap risk, flexibility value, and exit constraints."
        ),
    },
]

policy_df = pd.DataFrame(policy_docs)
policy_df


In [ ]:
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "sentence-transformers/all-MiniLM-L6-v2")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

policy_embeddings = embedder.encode(policy_df["text"].tolist(), normalize_embeddings=True)

def retrieve_policy(query: str, top_k: int = 1) -> List[Dict[str, Any]]:
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, policy_embeddings)[0]
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in ranked_indices:
        results.append({
            "policy_id": policy_df.iloc[idx]["policy_id"],
            "text": policy_df.iloc[idx]["text"],
            "score": float(scores[idx]),
        })
    return results


for case in synthetic_cases:
    retrieved = retrieve_policy(case["request"], top_k=1)[0]
    print(case["case_id"], "→", retrieved["policy_id"], "| score:", round(retrieved["score"], 3))


In [ ]:
case = synthetic_cases[1]
retrieved = retrieve_policy(case["request"], top_k=1)[0]

rag_prompt = structured_prompt(case, retrieved_policy=retrieved["text"])
rag_result = generate_hf(rag_prompt, max_new_tokens=220)

print("Retrieved policy:", retrieved["policy_id"])
print("-" * 100)
print(rag_result["output_text"])
print("-" * 100)
print({
    "input_tokens": rag_result["input_tokens"],
    "output_tokens": rag_result["output_tokens"],
    "latency_seconds": round(rag_result["latency_seconds"], 3),
})


## 5. Run evaluation across all synthetic cases

We evaluate three variants:

1. Basic prompt
2. Finance analyst prompt
3. RAG-grounded prompt

Metrics:

- ROUGE-L against a synthetic reference answer
- Presence of required finance sections
- Latency
- Input / output token count


In [ ]:
required_sections = [
    "Business Objective",
    "Compute Need",
    "Cost Drivers",
    "Risks",
    "Recommended Next Step",
]

def section_coverage_score(text: str) -> float:
    normalized = text.lower()
    hits = sum(1 for section in required_sections if section.lower() in normalized)
    return hits / len(required_sections)


def run_variant(case: Dict[str, Any], variant: str) -> Dict[str, Any]:
    if variant == "basic":
        prompt = basic_prompt(case)
        retrieved_policy_id = None
    elif variant == "finance_analyst":
        prompt = finance_analyst_prompt(case)
        retrieved_policy_id = None
    elif variant == "rag":
        retrieved = retrieve_policy(case["request"], top_k=1)[0]
        prompt = structured_prompt(case, retrieved_policy=retrieved["text"])
        retrieved_policy_id = retrieved["policy_id"]
    else:
        raise ValueError(f"Unknown variant: {variant}")
    
    result = generate_hf(prompt, max_new_tokens=220)
    
    return {
        "case_id": case["case_id"],
        "request_type": case["request_type"],
        "variant": variant,
        "retrieved_policy_id": retrieved_policy_id,
        "prompt": prompt,
        "output_text": result["output_text"],
        "reference_answer": case["reference_answer"],
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "latency_seconds": result["latency_seconds"],
        "section_coverage": section_coverage_score(result["output_text"]),
    }


rows = []
for case in synthetic_cases:
    for variant in ["basic", "finance_analyst", "rag"]:
        rows.append(run_variant(case, variant))

results_df = pd.DataFrame(rows)
results_df[[
    "case_id",
    "variant",
    "retrieved_policy_id",
    "input_tokens",
    "output_tokens",
    "latency_seconds",
    "section_coverage",
]]


In [ ]:
rouge = evaluate.load("rouge")

rouge_rows = []
for _, row in results_df.iterrows():
    scores = rouge.compute(
        predictions=[row["output_text"]],
        references=[row["reference_answer"]],
    )
    rouge_rows.append({
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"],
        "rougeLsum": scores["rougeLsum"],
    })

rouge_df = pd.DataFrame(rouge_rows)
results_scored_df = pd.concat([results_df.reset_index(drop=True), rouge_df], axis=1)

summary_metrics = (
    results_scored_df
    .groupby("variant")
    .agg(
        avg_rougeL=("rougeL", "mean"),
        avg_section_coverage=("section_coverage", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_latency_seconds=("latency_seconds", "mean"),
    )
    .reset_index()
)

summary_metrics


In [ ]:
pd.set_option("display.max_colwidth", 500)

results_scored_df[[
    "case_id",
    "variant",
    "output_text",
    "rougeL",
    "section_coverage",
]].head(10)


## 6. Token and cost estimation

This is a simple cost template. Replace the sample rates with the actual model pricing if you are using a paid API.

The point is not to estimate exact cost here. The point is to show a repeatable cost framework:

- input tokens
- output tokens
- requests
- model routing
- cache hit rate
- latency
- quality


In [ ]:
def estimate_token_cost(
    input_tokens: int,
    output_tokens: int,
    input_cost_per_1m: float,
    output_cost_per_1m: float,
) -> float:
    return (
        input_tokens / 1_000_000 * input_cost_per_1m
        + output_tokens / 1_000_000 * output_cost_per_1m
    )


# Example placeholder rates. Replace these with actual model pricing.
ASSUMED_INPUT_COST_PER_1M = 0.15
ASSUMED_OUTPUT_COST_PER_1M = 0.60

results_scored_df["estimated_cost_per_request"] = results_scored_df.apply(
    lambda r: estimate_token_cost(
        input_tokens=r["input_tokens"],
        output_tokens=r["output_tokens"],
        input_cost_per_1m=ASSUMED_INPUT_COST_PER_1M,
        output_cost_per_1m=ASSUMED_OUTPUT_COST_PER_1M,
    ),
    axis=1,
)

cost_summary = (
    results_scored_df
    .groupby("variant")
    .agg(
        avg_cost_per_request=("estimated_cost_per_request", "mean"),
        avg_latency_seconds=("latency_seconds", "mean"),
        avg_rougeL=("rougeL", "mean"),
        avg_section_coverage=("section_coverage", "mean"),
    )
    .reset_index()
)

cost_summary


## 7. Structured output validation

Small local models may not always follow strict JSON instructions.  
This section demonstrates a practical validation layer that checks whether the generated response includes the required business sections.


In [ ]:
def validate_decision_memo(text: str) -> Dict[str, Any]:
    present_sections = []
    missing_sections = []
    
    for section in required_sections:
        if section.lower() in text.lower():
            present_sections.append(section)
        else:
            missing_sections.append(section)
    
    return {
        "is_valid": len(missing_sections) == 0,
        "present_sections": present_sections,
        "missing_sections": missing_sections,
        "section_coverage": len(present_sections) / len(required_sections),
    }


validation_rows = []
for _, row in results_scored_df.iterrows():
    validation = validate_decision_memo(row["output_text"])
    validation_rows.append({
        "case_id": row["case_id"],
        "variant": row["variant"],
        **validation,
    })

validation_df = pd.DataFrame(validation_rows)
validation_df.head(10)


## 8. Optional OpenAI and Anthropic API wrappers

These cells are optional. They only run if you set API keys in your environment.

Do not commit API keys to GitHub.

Example environment variables:

- `OPENAI_API_KEY`
- `OPENAI_MODEL`
- `ANTHROPIC_API_KEY`
- `ANTHROPIC_MODEL`


In [ ]:
def call_openai_optional(prompt: str) -> Optional[str]:
    if not os.getenv("OPENAI_API_KEY"):
        return None
    
    from openai import OpenAI
    
    client = OpenAI()
    model_name = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
    
    response = client.responses.create(
        model=model_name,
        input=prompt,
    )
    
    return response.output_text


def call_anthropic_optional(prompt: str) -> Optional[str]:
    if not os.getenv("ANTHROPIC_API_KEY"):
        return None
    
    import anthropic
    
    client = anthropic.Anthropic()
    model_name = os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")
    
    response = client.messages.create(
        model=model_name,
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )
    
    return response.content[0].text


In [ ]:
# Optional API test.
# This will return None unless the relevant API key is set.

api_test_prompt = structured_prompt(synthetic_cases[1], retrieved_policy=retrieve_policy(synthetic_cases[1]["request"])[0]["text"])

openai_output = call_openai_optional(api_test_prompt)
anthropic_output = call_anthropic_optional(api_test_prompt)

print("OpenAI output available:", openai_output is not None)
print("Anthropic output available:", anthropic_output is not None)

if openai_output:
    print("\nOpenAI output:")
    print(openai_output)

if anthropic_output:
    print("\nAnthropic output:")
    print(anthropic_output)


## 9. PEFT / LoRA setup

This section shows how you would prepare a LoRA adapter for parameter-efficient fine-tuning.

The notebook does not run a full fine-tuning job by default because that would be heavier and requires more careful train / validation splitting.  
However, this setup demonstrates the right Hugging Face `peft` components.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

peft_model = get_peft_model(model, lora_config)

print("PEFT / LoRA model created.")
peft_model.print_trainable_parameters()


## 10. Executive summary

The final step is to turn evaluation results into a short executive summary.


In [ ]:
def generate_executive_summary(metrics_df: pd.DataFrame) -> str:
    best_quality = metrics_df.sort_values(
        ["avg_section_coverage", "avg_rougeL"],
        ascending=False,
    ).iloc[0]
    
    lowest_cost = metrics_df.sort_values("avg_cost_per_request").iloc[0]
    fastest = metrics_df.sort_values("avg_latency_seconds").iloc[0]
    
    summary = f"""
Executive Summary

Objective:
Evaluate prompt and retrieval strategies for AI infrastructure finance decision memos.

Key Findings:
- Best quality variant: {best_quality["variant"]}, with average section coverage of {best_quality["avg_section_coverage"]:.2f} and average ROUGE-L of {best_quality["avg_rougeL"]:.2f}.
- Lowest estimated token cost variant: {lowest_cost["variant"]}, with average estimated cost per request of ${lowest_cost["avg_cost_per_request"]:.8f}.
- Fastest variant: {fastest["variant"]}, with average latency of {fastest["avg_latency_seconds"]:.2f} seconds.

Recommendation:
Use retrieval-grounded prompting for higher-risk AI infrastructure finance decisions because it anchors the response in policy context and improves structure. 
For high-volume low-risk requests, use a smaller local or lower-cost model with routing, caching, and fallback to a stronger model when confidence or section coverage is low.

Next Steps:
1. Replace synthetic data with approved internal or public benchmark cases.
2. Add human evaluation for correctness, risk identification, and executive usefulness.
3. Compare local Hugging Face models against OpenAI and Anthropic models using the same evaluation harness.
4. Add production monitoring for latency, cost, and structured-output reliability.
"""
    return textwrap.dedent(summary).strip()


print(generate_executive_summary(cost_summary))


## 11. Suggested GitHub README text

You can copy this section into your repo README if needed.

Project title: LLM Infrastructure Finance Evaluation Assistant

Description:
An end-to-end LLM evaluation notebook for AI infrastructure finance requests. The project uses Hugging Face Transformers, Datasets, Evaluate, Sentence Transformers, and PEFT, with optional OpenAI and Anthropic API wrappers. It compares prompt strategies using synthetic finance cases, retrieval-grounded prompting, ROUGE scoring, structured-output validation, latency tracking, and token-cost estimation.

Why this matters:
AI infrastructure finance decisions require balancing model quality, compute capacity, latency, cost, safety, and roadmap uncertainty. This notebook demonstrates a practical evaluation workflow for those tradeoffs.
